# Multi-Company Metric Normalization (Slice 3B)

OwnerLens Feature 1 normalization began Adobe-only. This notebook shows how the same *economic* concepts are represented consistently across **Adobe (ADBE)**, **Visa (V)**, and **Costco (COST)** even though their SEC XBRL tags differ — and why some metrics are deliberately reported as *absent* or *unsupported* rather than guessed.

We answer eight questions:

1. What is a canonical financial metric?
2. Why can't OwnerLens assume every company uses the same XBRL tag?
3. Which mappings are common across ADBE, V, and COST?
4. Where do they differ?
5. What is a company override?
6. How is provenance preserved despite canonicalization?
7. Why is an explicit unsupported metric better than a guessed value?
8. What did Visa and Costco reveal about Adobe-shaped assumptions?

> Requires `OWNER_LENS_SEC_USER_AGENT` for the live SEC calls below.

In [ ]:
import os

from owner_lens import (
    ConceptNotFoundError,
    SecClient,
    normalize_annual_revenue,
    normalize_capital_expenditures,
    normalize_current_debt,
    normalize_diluted_shares,
    normalize_long_term_debt,
    normalize_short_term_investments,
    normalize_total_equity,
)
from owner_lens.metrics import (
    CURRENT_DEBT,
    LONG_TERM_DEBT,
    REVENUE,
    TOTAL_EQUITY,
    resolve_concepts,
)

USER_AGENT = os.environ.get('OWNER_LENS_SEC_USER_AGENT', 'OwnerLens admin@example.com')
TICKERS = ['ADBE', 'V', 'COST']

with SecClient(USER_AGENT) as client:
    facts = {t: client.retrieve_company_facts(t).raw_facts for t in TICKERS}

list(facts)

## 1. What is a canonical financial metric?

A canonical metric describes the *economic concept* OwnerLens wants — independent of any one company. It names the concept, its fact kind (duration or instant), its expected unit, an ordered default source-concept preference, and any minimal per-company overrides.

In [ ]:
print('name          :', REVENUE.name)
print('kind          :', REVENUE.kind.value)
print('unit          :', REVENUE.unit)
print('default concepts:')
for concept in REVENUE.default_concepts:
    print('   ', concept)

## 2. Why can't OwnerLens assume every company uses the same XBRL tag?

The same economic idea — revenue — is tagged differently by different filers. Adobe reports `Revenues`; Visa and Costco report `RevenueFromContractWithCustomerExcludingAssessedTax`. A single hard-coded tag would silently fail or mis-select.

In [ ]:
for t in TICKERS:
    series = normalize_annual_revenue(facts[t], ticker=t)
    print(f'{t:5} revenue concept -> {series.concept}')

## 3 & 4. Which mappings are common, and where do they differ?

Most metrics resolve through shared default preferences. A few differ: Visa's CapEx uses `PaymentsToAcquireProductiveAssets`; Visa and Costco express debt through the current/noncurrent split; Visa's equity includes noncontrolling interest; and Visa reports no diluted-share concept at all.

In [ ]:
metrics = [
    ('Revenue', normalize_annual_revenue),
    ('CapEx', normalize_capital_expenditures),
    ('Current Debt', normalize_current_debt),
    ('Long-Term Debt', normalize_long_term_debt),
    ('Equity', normalize_total_equity),
    ('Diluted Shares', normalize_diluted_shares),
]


def source_concept(fn, ticker):
    try:
        series = fn(facts[ticker], ticker=ticker)
    except ConceptNotFoundError:
        return 'unsupported'
    return series.concept or 'absent'


for label, fn in metrics:
    cells = '   '.join(f'{t}={source_concept(fn, t)}' for t in TICKERS)
    print(f'{label:16} {cells}')

## 5. What is a company override?

An override is explicit, minimal, declarative data scoped to one canonical metric and one company. It *replaces* the default concept preference for that company (it does not append), because Adobe and Visa/Costco have opposite stale tags — reordering the default would regress Adobe. Adobe carries no overrides.

In [ ]:
for definition in (CURRENT_DEBT, LONG_TERM_DEBT, TOTAL_EQUITY):
    print(definition.name)
    for t in TICKERS:
        print(f'   {t:5} -> {resolve_concepts(definition, t)}')

## 6. How is provenance preserved despite canonicalization?

Canonicalization never hides the source. Every normalized series records the *actual selected source concept* alongside the canonical metric name and company, so a reviewer or agent can always trace the value back.

In [ ]:
v_equity = normalize_total_equity(facts['V'], ticker='V')
print('canonical metric:', v_equity.metric)
print('source concept  :', v_equity.concept)
print('company         :', v_equity.ticker)

## 7. Why is an explicit unsupported metric better than a guessed value?

OwnerLens keeps three outcomes distinct: **absent** (structural or policy), **unsupported** (no trustworthy concept), and **reported zero** (a real observation). Visa tags no weighted-average diluted-share count, so diluted shares is *unsupported* — never derived from EPS. Visa's investment securities are *absent by policy* for short-term investments, not absorbed as corporate cash.

> `absent` ≠ `unsupported` ≠ `zero`

In [ ]:
try:
    normalize_diluted_shares(facts['V'], ticker='V')
except ConceptNotFoundError:
    print('Visa diluted shares -> UNSUPPORTED (no trustworthy concept); not guessed')

v_sti = normalize_short_term_investments(facts['V'], ticker='V')
print('Visa short-term investments observations:', v_sti.observations)
print('-> absent by canonical policy (investment securities are not corporate cash)')

## 8. What did Visa and Costco reveal about Adobe-shaped assumptions?

- **Revenue tag** is not universal: Adobe's `Revenues` is not what Visa or Costco use.
- **CapEx tag** differs: Visa uses `PaymentsToAcquireProductiveAssets`.
- **Debt structure** differs: Visa and Costco split current/noncurrent (`LongTermDebtCurrent` + `LongTermDebtNoncurrent`), while Adobe uses `DebtCurrent` + `LongTermDebt`. The correct pairing avoids double counting.
- **Equity tag** differs: Visa reports equity including noncontrolling interest.
- **Some metrics simply do not exist** for a company (Visa diluted shares), and some financial assets must *not* be reinterpreted as corporate cash (Visa short-term investments).

The result is a small, auditable registry of canonical definitions plus minimal, traceable overrides — not a pile of ticker-specific hacks. This is the normalization foundation for Slice 3C's full golden-company validation.

*This notebook does not perform investment analysis; it demonstrates canonical normalization and provenance only.*